In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import gpytorch
import torch
import matplotlib.pyplot as plt
# with pd.ExcelFile("datasets/stresstool_update.xls") as xls:
#     df0 = pd.read_excel(xls, "UserInterface")
#     df1 = pd.read_excel(xls, "Kriging")
#     df2 = pd.read_excel(xls, "Quadratic")

In [62]:
df_data = pd.read_csv("datasets/Kriging_data.csv", header=[0, 1])
df_data.columns

MultiIndex([(        't_lf', 'Unnamed: 0_level_1'),
            (    't_solder', 'Unnamed: 1_level_1'),
            ('MidDieStress',               'top2'),
            ('MidDieStress',               'top5'),
            ('MidDieStress',               'bot4'),
            ('MidDieStress',               'bot5'),
            ('MaxDieStress',               'top2'),
            ('MaxDieStress',               'top5'),
            ('MaxDieStress',               'bot4'),
            ('MaxDieStress',               'bot5')],
           )

In [63]:
X_bounds = np.array([[.2, 1.], [.02, .08]])

In [64]:
X = df_data[["t_lf", "t_solder"]].values

In [65]:
X_scaled = torch.tensor((X - X_bounds[:, 0]) / (X_bounds[:, 1] - X_bounds[:, 0]))

In [66]:
k = 1
N = len(X_scaled)
# for i in range(N // k):
#     plt.scatter(X_scaled[:, 0], X_scaled[:, 1])
#     plt.scatter(X_scaled[k * i:k * (i + 1), 0], X_scaled[k * i:k * (i + 1), 1])
#     plt.xlim(-.05, 1.05)
#     plt.ylim(-.05, 1.05)
#     plt.show()

In [67]:
y = np.abs(df_data[('MidDieStress', 'top2')].values[:, None])
y

array([[23926.825022],
       [ 8673.64329 ],
       [  550.882608],
       [ 2899.288678],
       [ 2946.433399],
       [ 7561.378932],
       [ 3922.703139],
       [  608.634532],
       [  655.532497],
       [12983.097303],
       [ 8245.748616],
       [  463.168584],
       [  753.04254 ],
       [ 6011.239726],
       [  885.012646],
       [12241.338842],
       [ 9311.823487],
       [ 5407.810277],
       [ 2485.45288 ],
       [ 9408.70384 ],
       [ 2507.390278],
       [17864.950155],
       [  487.438467],
       [ 2905.100404],
       [11766.761307]])

In [68]:
scaler = StandardScaler()
y_scaled = torch.tensor(scaler.fit_transform(y))
y_scaled

tensor([[ 2.9865],
        [ 0.4140],
        [-0.9559],
        [-0.5599],
        [-0.5519],
        [ 0.2264],
        [-0.3873],
        [-0.9462],
        [-0.9383],
        [ 1.1408],
        [ 0.3418],
        [-0.9707],
        [-0.9218],
        [-0.0350],
        [-0.8996],
        [ 1.0157],
        [ 0.5216],
        [-0.1368],
        [-0.6297],
        [ 0.5380],
        [-0.6260],
        [ 1.9641],
        [-0.9666],
        [-0.5589],
        [ 0.9357]], dtype=torch.float64)

In [69]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.MaternKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

def train(X_scaled, y_scaled):
    # initialize likelihood and model
    # likelihood = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=gpytorch.constraints.Interval(1e-4, 1e-3))
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = ExactGPModel(X_scaled, y_scaled.flatten(), likelihood)
    training_iter = 150

    # Find optimal model hyperparameters
    model.train()
    likelihood.train()

    # Use the adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(training_iter):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model(X_scaled)
        # Calc loss and backprop gradients
        loss = -mll(output, y_scaled.flatten())
        loss.backward()
        print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
            i + 1, training_iter, loss.item(),
            model.covar_module.base_kernel.lengthscale.item(),
            model.likelihood.noise.item()
        ))
        optimizer.step()

        # print(list(model.named_parameters()))
    
    return model, likelihood

In [70]:
model_likelihood_list = []
for i in range(N // k):
    test_idx = np.arange(k * i, k * (i + 1))
    train_idx = np.delete(np.arange(N), test_idx)
    model, likelihood = train(X_scaled=X_scaled[train_idx], y_scaled=y_scaled[train_idx])
    model_likelihood_list.append((model, likelihood))

Iter 1/150 - Loss: 1.191   lengthscale: 0.693   noise: 0.693
Iter 2/150 - Loss: 1.178   lengthscale: 0.744   noise: 0.644
Iter 3/150 - Loss: 1.165   lengthscale: 0.725   noise: 0.598
Iter 4/150 - Loss: 1.154   lengthscale: 0.690   noise: 0.555
Iter 5/150 - Loss: 1.145   lengthscale: 0.650   noise: 0.515
Iter 6/150 - Loss: 1.136   lengthscale: 0.609   noise: 0.477
Iter 7/150 - Loss: 1.129   lengthscale: 0.568   noise: 0.443
Iter 8/150 - Loss: 1.124   lengthscale: 0.528   noise: 0.411
Iter 9/150 - Loss: 1.120   lengthscale: 0.490   noise: 0.383
Iter 10/150 - Loss: 1.118   lengthscale: 0.453   noise: 0.359
Iter 11/150 - Loss: 1.117   lengthscale: 0.418   noise: 0.337
Iter 12/150 - Loss: 1.117   lengthscale: 0.387   noise: 0.319
Iter 13/150 - Loss: 1.119   lengthscale: 0.361   noise: 0.304
Iter 14/150 - Loss: 1.120   lengthscale: 0.343   noise: 0.292
Iter 15/150 - Loss: 1.122   lengthscale: 0.332   noise: 0.283
Iter 16/150 - Loss: 1.123   lengthscale: 0.329   noise: 0.277
Iter 17/150 - Los

In [71]:
x_grid = torch.linspace(0, 1, 50)
test_x = torch.stack(torch.meshgrid(x_grid, x_grid)).reshape(2, -1).T

In [72]:
for i, (model, likelihood) in enumerate(model_likelihood_list):
    test_idx = np.arange(k * i, k * (i + 1))
    X_test = X_scaled[test_idx]
    
    train_idx = np.delete(np.arange(N), test_idx)

    # Get into evaluation (predictive posterior) mode
    model.eval()
    likelihood.eval()
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(X_test))
    print(y_scaled[test_idx].flatten(), observed_pred.mean)

tensor([2.9865], dtype=torch.float64) tensor([0.6857], dtype=torch.float64)
tensor([0.4140], dtype=torch.float64) tensor([-0.2254], dtype=torch.float64)
tensor([-0.9559], dtype=torch.float64) tensor([1.4474], dtype=torch.float64)
tensor([-0.5599], dtype=torch.float64) tensor([-0.2236], dtype=torch.float64)
tensor([-0.5519], dtype=torch.float64) tensor([-0.1136], dtype=torch.float64)
tensor([0.2264], dtype=torch.float64) tensor([0.4440], dtype=torch.float64)
tensor([-0.3873], dtype=torch.float64) tensor([-0.4748], dtype=torch.float64)
tensor([-0.9462], dtype=torch.float64) tensor([-0.4946], dtype=torch.float64)
tensor([-0.9383], dtype=torch.float64) tensor([-0.5265], dtype=torch.float64)
tensor([1.1408], dtype=torch.float64) tensor([0.9501], dtype=torch.float64)
tensor([0.3418], dtype=torch.float64) tensor([0.0380], dtype=torch.float64)
tensor([-0.9707], dtype=torch.float64) tensor([-0.6395], dtype=torch.float64)
tensor([-0.9218], dtype=torch.float64) tensor([-0.6683], dtype=torch.float

In [78]:
for i, (model, likelihood) in enumerate(model_likelihood_list):
    # Get into evaluation (predictive posterior) mode
    model.eval()
    likelihood.eval()
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(X_scaled))
    # print(scaler.inverse_transform(y_scaled).flatten())
    print(scaler.inverse_transform(observed_pred.mean[:, None]).flatten())
    print()

[10284.83540111  6134.03614806  7077.47273478  4590.39374318
  4755.53905918  7741.02209587  3588.62938069  2231.70403673
  1746.2949164  11002.70698051  7032.09935184  1368.23363703
  1988.72426302  7739.40776435  6572.95581043  7426.87209388
  5300.3618992   4500.37760993  1480.92917751  7225.83963787
  4304.67639989 12087.51406398  2660.81626591  5113.01597117
  8691.65378189]

[14717.40354711  4882.33652686  7708.7689999   4204.21185166
  4761.67870295  8487.31542531  3431.26802023  2268.50041683
  1967.4376871  12400.74766942  6631.77803569  1391.47986965
  1684.06290348  7602.63510004  7139.37721886  7460.14263885
  5479.93510665  4335.70593992  1586.64172269  7329.61543061
  3736.43464877 15196.17582102  2204.9524588   4319.26950773
 11295.63891342]

[16600.94072886  5964.6052526  14800.98034822  4125.32124537
  4367.67865347  9618.60664527  3300.22665787  1822.68962008
  1005.78856995 13090.17280745  6696.5469511    341.69744097
  1426.71964164  9025.37471459  7345.05027874 107